# Capítulo 2 — Arquitetura Transformer: Por Dentro de um LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap02_transformers.ipynb)

**Fonte original:** [orca3/llm-model-serving — ch02/ch2_Inside_the_Mind_of_a_Transformer.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch02) e [ch02/ch2_Workthrough_LLM_execution.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch02).

Adaptado e comentado por **Anderson Ejepsen**.

Créditos aos autores originais do repositório [orca3/llm-model-serving](https://github.com/orca3/llm-model-serving).

## Configuração do Ambiente

Instalação das dependências necessárias para explorar a arquitetura Transformer.

> **NOTA:** Recomendamos usar GPU para executar este notebook. No Google Colab, vá em Runtime > Change runtime type > Hardware accelerator > GPU > T4.

In [ ]:
!pip install --quiet transformers tiktoken transformers_stream_generator bertviz matplotlib

## Função utilitária para liberar memória GPU

Como vamos carregar e descarregar modelos várias vezes, precisamos de uma função que libere a memória da GPU corretamente. Ela remove a referência ao modelo, limpa o cache CUDA e força a coleta de lixo.

In [ ]:
import torch
import gc
import time

def free_gpu(model):
    """Descarrega o modelo e libera a memória da GPU."""
    if model:
        # Remove a referência ao modelo, tornando-o elegível para coleta de lixo
        del model

    # Libera memória CUDA em cache que não está mais em uso
    torch.cuda.empty_cache()

    # Força a coleta de lixo para garantir liberação completa da memória
    gc.collect()

---
## 2.1 — Configuração do Modelo (model.config)

O objeto `model.config` contém todos os hiperparâmetros e configurações do modelo. Ele define a arquitetura, o tamanho e o comportamento — é essencialmente o **blueprint** de como o modelo é estruturado.

Vamos carregar o Qwen2.5-0.5B e inspecionar seus parâmetros.

In [ ]:
from transformers import AutoModelForCausalLM
from pprint import pprint

model_name = "Qwen/Qwen2.5-0.5B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    device_map="auto"
)

# Imprime todos os parâmetros de configuração
config = model.config
print("\n=== Parâmetros de Configuração do Modelo ===")

# Parâmetros de arquitetura
print("\nParâmetros de Arquitetura:")
print(f"Tamanho da camada oculta (hidden_size): {config.hidden_size}")
print(f"Número de camadas: {config.num_hidden_layers}")
print(f"Número de cabeças de atenção: {config.num_attention_heads}")
print(f"Tamanho intermediário (MLP): {config.intermediate_size}")

# Parâmetros do tokenizador
print("\nParâmetros do Tokenizador:")
print(f"Tamanho do vocabulário: {config.vocab_size}")
print(f"Posições máximas (embeddings): {config.max_position_embeddings}")

# Tamanho total do modelo
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTamanho do Modelo:")
print(f"Total de parâmetros: {total_params:,}")

# Parâmetros específicos do modelo
print("\nParâmetros específicos:")
for key, value in config.to_dict().items():
    if key not in ['architectures', 'model_type', 'torch_dtype']:
        print(f"{key}: {value}")

# Libera memória GPU
free_gpu(model)

---
## 2.2 — Arquitetura do Modelo

Agora vamos visualizar a estrutura interna do modelo — as camadas de embedding, os blocos Transformer e as projeções de atenção. Isso nos permite entender como os dados fluem através da rede.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from pprint import pprint

# Carrega o modelo
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    trust_remote_code=True,
    device_map="auto"
)

print(f"\n=== Arquitetura do {model_name} ===")
print("\nEstrutura do Modelo:")

def print_module_structure(module, prefix=''):
    """Imprime a estrutura hierárquica do modelo recursivamente."""
    for name, child in module.named_children():
        if name in ['_orig_mod', 'wrapped_model']:
            continue

        print(f"{prefix}{name}: {type(child).__name__}")

        if "Qwen2Attention" in name.lower():
            print(f"\nMódulo de atenção encontrado: {name}")
            print(f"Tipo: {type(module).__name__}")

            if hasattr(module, 'num_heads'):
                print(f"Número de cabeças de atenção: {module.num_heads}")
            if hasattr(module, 'head_dim'):
                print(f"Dimensão por cabeça: {module.head_dim}")
            if hasattr(module, 'hidden_size'):
                print(f"Tamanho oculto: {module.hidden_size}")
            if hasattr(module, 'rotary_emb'):
                print(f"Possui embeddings rotatórios (RoPE): {module.rotary_emb is not None}")

        if list(child.children()):
            print_module_structure(child, prefix + '  ')

print_module_structure(model)

free_gpu(model)

---
## 2.3 — Análise Detalhada das Camadas de Atenção

A atenção é o mecanismo central dos Transformers. Vamos inspecionar cada camada de atenção, suas projeções Q/K/V e os atributos específicos.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    trust_remote_code=True,
    device_map="auto"
)

print(f"\n=== Análise Detalhada de Atenção do {model_name} ===")

# Encontra todas as camadas de atenção
attention_layers = []
for name, module in model.named_modules():
    if "Qwen2Attention" in type(module).__name__:
        attention_layers.append((name, module))

print(f"\nEncontradas {len(attention_layers)} camadas de atenção")

# Analisa cada camada de atenção
for i, (name, module) in enumerate(attention_layers):
    print(f"\nCamada de Atenção {i}: {name}")
    print("=" * 50)

    print(f"Tipo: {type(module).__name__}")

    if hasattr(module, 'num_heads'):
        print(f"Número de cabeças de atenção: {module.num_heads}")
    if hasattr(module, 'head_dim'):
        print(f"Dimensão por cabeça: {module.head_dim}")
    if hasattr(module, 'hidden_size'):
        print(f"Tamanho oculto: {module.hidden_size}")
    if hasattr(module, 'rotary_emb'):
        print(f"Embeddings rotatórios: {type(module.rotary_emb).__name__ if module.rotary_emb else 'Nenhum'}")

    # Projeções de atenção (Q, K, V, Output)
    print("\nProjeções de atenção:")
    for sub_name, sub_module in module.named_children():
        if hasattr(sub_module, 'weight'):
            shape = sub_module.weight.shape
            print(f"  {sub_name}: {type(sub_module).__name__}, Shape: {shape}")

    # Atributos adicionais específicos de atenção
    print("\nAtributos adicionais:")
    for attr_name in dir(module):
        if not attr_name.startswith('_') and not callable(getattr(module, attr_name)):
            try:
                value = getattr(module, attr_name)
                if not isinstance(value, (torch.Tensor, torch.nn.Module)):
                    print(f"  {attr_name}: {value}")
            except:
                pass

# Configuração relacionada à atenção
print("\nConfiguração de atenção:")
config = model.config.to_dict()
attention_config = {k: v for k, v in config.items() if 'attention' in k.lower()}
pprint(attention_config)

free_gpu(model)

---
## 2.4 — Cálculo de Memória do Modelo

Antes de fazer deploy de um LLM, é fundamental estimar quanta memória ele vai consumir. Esta função calcula a memória necessária com base nos hiperparâmetros do Qwen2.5-0.5B, considerando embeddings, projeções Q/K/V, MLP e normalizações.

In [ ]:
def calculate_model_memory():
    """Calcula a memória necessária para o modelo Qwen2.5-0.5B em float32."""
    # Constantes da arquitetura
    hidden_size = 896
    num_layers = 24
    num_heads = 14
    intermediate_size = 4864
    vocab_size = 151936
    max_position = 32768

    # Memória para embeddings
    embedding_memory = vocab_size * hidden_size * 4  # 4 bytes por float32

    # Memória por bloco Transformer
    # Self-attention: projeções Q, K, V
    qkv_memory = hidden_size * hidden_size * 3 * 4
    attention_output_memory = hidden_size * hidden_size * 4  # Projeção de saída

    # MLP
    mlp_input_memory = hidden_size * intermediate_size * 4  # Primeira camada MLP
    mlp_output_memory = intermediate_size * hidden_size * 4  # Segunda camada MLP

    # Normalizações de camada
    norm_memory = hidden_size * 4

    # Memória total por camada
    layer_memory = (qkv_memory + attention_output_memory +
                   mlp_input_memory + mlp_output_memory +
                   norm_memory * 2)  # 2 normalizações por bloco

    # Memória total do modelo
    total_memory = (embedding_memory + layer_memory * num_layers)

    # Converte para MB
    total_memory_mb = total_memory / (1024 * 1024)

    return {
        "Memória de Embedding (MB)": embedding_memory / (1024 * 1024),
        "Memória por Camada (MB)": layer_memory / (1024 * 1024),
        "Memória Total do Modelo (MB)": total_memory_mb,
        "Memória Total do Modelo (GB)": total_memory_mb / 1024
    }

# Calcula e imprime o uso de memória
memory_stats = calculate_model_memory()
print("\n=== Análise de Uso de Memória ===")
for key, value in memory_stats.items():
    print(f"{key}: {value:.2f}")

---
## 2.5 — Visualização da Atenção com BertViz

Usando a biblioteca `bertviz`, podemos visualizar como o modelo distribui atenção entre os tokens de entrada. Isso é fundamental para entender **quais palavras influenciam a predição de cada token**.

In [ ]:
from transformers import AutoTokenizer
from bertviz import head_view

# Texto de entrada para visualização
text = "The tiny animal was overwhelmed by the confetti and it attempted to bat away the glitter with its little paws."
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, output_attentions=True)
model = AutoModelForCausalLM.from_pretrained(model_name, output_attentions=True).eval().cuda()

# Tokeniza a entrada e obtém os tokens como strings
inputs = tokenizer(text, return_tensors="pt").to(model.device)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# Gera as saídas com pesos de atenção
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

# Obtém os pesos de atenção
attention = outputs.attentions

# Visualiza a atenção com bertviz
head_view(attention, tokens)

---
## 2.6 — Execução Passo a Passo de um LLM

Agora vamos decompor o processo de geração de texto para entender cada etapa:
1. Tokenização do prompt
2. Forward pass pelo modelo
3. Seleção do próximo token (sampling)
4. Loop autorregressivo até EOS

Isso é fundamental para entender **por que a geração do primeiro token é mais lenta** (prefill) e os subsequentes são mais rápidos (decode).

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

# Inicializa o pipeline de geração de texto
generator = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B')

# Define o prompt
prompt = "Write a short introduction about US capital city."

# Gera texto
generated_text = generator(prompt, max_length=50, num_return_sequences=1)

# Imprime o texto gerado
print(generated_text[0]['generated_text'])

free_gpu(generator.model)

### Decompondo o loop de geração token a token

Abaixo implementamos o loop autorregressivo manualmente, medindo o tempo de cada token. Observe que:
- **(A)** Definimos o contexto atual
- **(B)** Geramos predições (logits) para o próximo token
- **(C)** Selecionamos o próximo token via softmax + multinomial sampling
- **(D)** Concatenamos o novo token à sequência
- **(E)** Verificamos se o token de fim de sequência (EOS) foi gerado

In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

first_token_generated = False

# (1) Especifica o modelo e carrega o tokenizador e o modelo
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).to("cuda" if torch.cuda.is_available() else "cpu")

# (2) Define o prompt de entrada
prompt = """The history of human communication is a story of innovation. From ancient cave paintings and spoken language to the invention of writing systems, humans have constantly developed new methods to express ideas and share knowledge. The printing press revolutionized the spread of information, enabling books to be produced and distributed at an unprecedented scale. Centuries later, the invention of the telegraph, radio, and television further transformed how we connect with one another. But perhaps no advancement has reshaped communication more profoundly than the internet.
Today, digital platforms allow billions of people to share messages, media, and experiences in real time. Social media, messaging apps, and video conferencing have broken down geographical barriers and created new ways of building communities. At the same time, these technologies raise important questions about privacy, information overload, and the nature of human interaction.
Looking ahead, emerging technologies such as virtual reality, brain-computer interfaces, and artificial intelligence promise to once again redefine how we communicate. As we reflect on this history and anticipate the future, one question arises:

How might the next wave of communication tools shape our relationships, societies, and sense of identity?"""

# (3) Tokeniza o prompt para o formato que o modelo entende
max_new_tokens = 100
idx = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
start_time = total_time = time.time()
times = []

# (4) Loop principal de geração — gera tokens um a um
for _ in range(max_new_tokens):

    # (A) Define o contexto atual para geração
    idx_cond = idx
    with torch.no_grad():
        # (B) Gera predições (candidatos de token) para o próximo token
        outputs = model(idx_cond)
        # Obtém os logits (scores brutos de predição) para cada token
        logits = outputs.logits

    # (C) Seleciona o próximo token a partir das predições
    logits = logits[:, -1, :]  # Seleciona apenas os logits do último token
    probas = torch.softmax(logits, dim=-1)  # Converte logits em probabilidades
    # Amostra o próximo token da distribuição de probabilidade
    idx_next = torch.multinomial(probas, num_samples=1)
    print("Próximo Token:", tokenizer.decode(idx_next[0], skip_special_tokens=True))
    time_cost = time.time() - start_time
    times.append(time_cost)

    # Rastreia o tempo gasto na geração de cada token
    if not first_token_generated:
        print(f"Tempo para gerar o primeiro token: {time_cost:.4f} segundos")
        first_token_generated = True
    else:
        print(f"Tempo para gerar um token: {time_cost:.4f} segundos")

    start_time = time.time()

    # (D) Adiciona o novo token à sequência de entrada
    idx = torch.cat((idx, idx_next), dim=1)

    # (E) Verifica se o token de fim de sequência (EOS) foi gerado
    if idx_next.item() == tokenizer.eos_token_id:
        print("\n[Geração concluída — token EOS atingido]")
        break

# Decodifica a sequência gerada completa
generated_text = tokenizer.decode(idx[0], skip_special_tokens=True)
print(f"Tempo total de geração: {time.time() - total_time:.4f} segundos")
print(generated_text)

# Libera memória GPU
free_gpu(model=model)

### Visualização do tempo de geração por token

O gráfico abaixo mostra o tempo de geração de cada token. Note que o **primeiro token (vermelho)** demora significativamente mais — isso é a fase de **prefill**, onde todos os tokens do prompt são processados de uma vez.

In [ ]:
import matplotlib.pyplot as plt

# Primeiro token em vermelho (prefill), demais em azul (decode)
plt.figure(figsize=(12, 4))
plt.bar(range(len(times)), times, color=['red'] + ['blue'] * (len(times) - 1))
plt.xlabel("ID do Token")
plt.ylabel("Tempo Gasto na Geração (s)")
plt.title("Tempo de Geração do LLM por Token")
plt.show()

---
## 2.7 — KV Cache: Acelerando a Geração

Sem KV Cache, a cada novo token gerado o modelo reprocessa **todos os tokens anteriores**. Com KV Cache, armazenamos os vetores Key e Value de iterações anteriores, e na nova iteração passamos **apenas o último token** como entrada. Isso reduz drasticamente o custo computacional.

Comparação:
- **Sem cache**: complexidade O(n²) por token
- **Com cache**: complexidade O(n) por token

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, use_cache=True) \
            .to("cuda" if torch.cuda.is_available() else "cpu")

prompt = """The history of human communication is a story of innovation. From ancient cave paintings and spoken language to the invention of writing systems, humans have constantly developed new methods to express ideas and share knowledge. The printing press revolutionized the spread of information, enabling books to be produced and distributed at an unprecedented scale. Centuries later, the invention of the telegraph, radio, and television further transformed how we connect with one another. But perhaps no advancement has reshaped communication more profoundly than the internet.
Today, digital platforms allow billions of people to share messages, media, and experiences in real time. Social media, messaging apps, and video conferencing have broken down geographical barriers and created new ways of building communities. At the same time, these technologies raise important questions about privacy, information overload, and the nature of human interaction.
Looking ahead, emerging technologies such as virtual reality, brain-computer interfaces, and artificial intelligence promise to once again redefine how we communicate. As we reflect on this history and anticipate the future, one question arises:

How might the next wave of communication tools shape our relationships, societies, and sense of identity?"""

num_interations = 100
times_with_cache = []

first_token_generated = False
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
idx = input_ids
start_time = total_time = time.time()

# (1) Define o KV Cache para geração mais rápida
past_key_values = None

for _ in range(num_interations):
    print("Tamanho de input_ids: " + str(input_ids.size()))
    with torch.no_grad():
        outputs = model(input_ids=input_ids,
          past_key_values=past_key_values,  # (2) Usa KV-cache da iteração anterior
          use_cache=True,                    # (2) Habilita armazenamento do KV cache
          max_new_tokens=100,
          min_new_tokens=100)

        logits = outputs.logits
        # (3) Atualiza o KV Cache
        past_key_values = outputs.past_key_values
        torch.cuda.synchronize()

    logits = logits[:, -1, :]
    probas = torch.softmax(logits, dim=-1)
    generated_token_id = torch.multinomial(probas, num_samples=1)

    # (4) Atualiza input_ids apenas com o novo token (usando KV-cache)
    input_ids = generated_token_id  # Nota: Não concatena com tokens anteriores devido ao KV-cache

    print("Próximo token:", tokenizer.decode(generated_token_id[0], skip_special_tokens=True))
    idx = torch.cat((idx, generated_token_id), dim=1)

    time_cost = time.time() - start_time
    times_with_cache.append(time_cost)
    if not first_token_generated:
        print(f"Tempo para o primeiro token: {time_cost:.4f} segundos")
        first_token_generated = True
    else:
        print(f"Tempo para o próximo token: {time_cost:.4f} segundos")
    start_time = time.time()

    if generated_token_id.item() == tokenizer.eos_token_id:
        print("\n[Geração concluída — token EOS atingido]")
        break

generated_text = tokenizer.decode(idx[0], skip_special_tokens=True)
print(f"Tempo total de geração: {time.time() - total_time:.4f} segundos")
print(generated_text)

free_gpu(model=model)

### Comparação: Sem Cache vs Com Cache

Os gráficos abaixo comparam o tempo de geração por token **sem** e **com** KV Cache. A diferença é dramática, especialmente para sequências longas.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

# Tempo de geração SEM KV Cache
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.bar(range(len(times)), times, color=['red'] + ['blue'] * (len(times) - 1))
plt.xlabel("ID do Token")
plt.ylabel("Tempo de Geração (s)")
plt.title("Geração SEM KV Cache")

# Tempo de geração COM KV Cache
plt.subplot(1, 2, 2)
plt.bar(range(len(times_with_cache)), times_with_cache, color=['red'] + ['blue'] * (len(times_with_cache) - 1))
plt.xlabel("ID do Token")
plt.ylabel("Tempo de Geração (s)")
plt.title("Geração COM KV Cache")

plt.tight_layout()
plt.show()